# CWE classification: a reproducible comparison

Compare ranking and traversal on the **same examples**, with a transparent lexical reference and optional OpenAI Decisions or legacy Jev selectors. The protocol defaults to `dev`, makes no remote calls unless explicitly enabled, and writes a resumable run with provenance. Gold CWEs are curator annotations, not independently adjudicated truth. This is top-1 suggestion evaluation with potentially multiple acceptable labels, not full multilabel set prediction.

Prepare a snapshot from the repository root: `uv run python -m experiment.fetch_sources --revision <full-commit-sha>`. Inspect `dataset_analysis.ipynb` first. Start Jupyter from the repository root with `uv run --extra notebook jupyter lab` and select the project kernel.

The previous test split was used to choose option wording and beam width. Treat those historical results as exploratory; obtain a new time-based or independently reviewed holdout before making a confirmation claim. Hypotheses, including bypassing roots, belong in `hypotheses.ipynb`.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

from classification import CWEGraph, normalize_label
from experiment import default_config
from experiment.runner import run_experiment

pd.set_option("display.precision", 3)

## Freeze the protocol before evaluating

1. Develop on `dev`, fixing model, prompt, traversal, sample size, metrics and stopping rules.
2. Set `RUN_REMOTE = True` only when ready to send benchmark text to the selected provider. The environment supplies `OPENAI_API_KEY`; credentials never belong in cells.
3. Freeze a configuration, then evaluate a fresh holdout once. Repeated dev comparisons are exploratory.

Flat ranks the entire catalog. Greedy follows one branch; beam retains several paths. Selectors offer the current CWE alongside its children as a stopping option. Flat Decisions requests are skipped when the catalog exceeds its 255-choice limit; do not silently compare a candidate shortlist with full-catalog flat ranking. One-option edges are deterministic and make no API request.

Beam multiplies local choice probabilities and keeps the best path to a DAG node. This favors short routes and is a heuristic: scores are not calibrated label probabilities and do not sum alternate paths. Threshold/margin settings apply only to scorers, not selectors. Pin a provider model where a snapshot is available; model aliases can change even if the returned identifier does not.

In [ ]:
config = default_config(Path.cwd())
config.update(split="dev", max_examples=100, seed=0)
RUN_REMOTE = False
if RUN_REMOTE:
    config["models"]["openai"] = "experiment.decisions_selector:create_selector"
    config["model_options"]["openai"] = {
        "model": "gpt-6-luna",
        "stop_style": "parent",
        "timeout": 30.0,
    }

# Optional baselines: add vector only after rebuilding a versioned index.
# config["models"]["vector"] = "vector"
# Legacy Jev requires `uv sync --extra jev --extra notebook`.
# config["models"]["jev"] = "experiment.jev_selector:create_parent_selector"
config

## Run and inspect provenance

A seeded shuffle selects a nested sample shared by every system. Each completed prediction is durably checkpointed. Matching checkpoints survive interrupted runs and smaller subsequent reports; changed data, catalog, code, dependencies, prompt metadata or options select a different checkpoint. A local lock prevents overlapping writers in one output directory. Refusals and malformed responses stop the run instead of silently disappearing from the denominator.

Predictions include source text and full decision traces. Keep output directories private when using nonpublic inputs. `metrics.json` records source hashes, dependency versions, exact selected IDs and catalog/benchmark hashes. Zero retries make request accounting explicit. Reports describe original prediction latency, including reused records, not the wall time of a resumed run.

In [ ]:
summary = run_experiment(config)
prediction_records = {
    system: [
        json.loads(line)
        for line in (Path(config["output"]) / f"{system.replace('/', '-')}.jsonl")
        .read_text(encoding="utf-8")
        .splitlines()
        if line.strip()
    ]
    for system in summary["systems"]
}
display(pd.Series(summary["examples"], name="sample").to_frame())
display(pd.Series(summary["reused"], name="reused predictions").to_frame())
pd.Series(summary["skipped"], name="skipped (reason)", dtype="object").to_frame()

### Helpers

Shared hierarchy helpers used by the analysis below.

In [ ]:
from functools import lru_cache

graph = CWEGraph(config["catalog"], config["view"])


def split_system(system):
    model, method = system.split("/")
    return {"model": model, "method": method}


def cwe_name(label):
    if label == "STOP":
        return label
    return f"CWE-{label} {graph.labels.get(label, {}).get('Name', 'outside catalog')}"


@lru_cache(None)
def ancestors(label):
    """A label and its ancestors; avoid enumerating every DAG path."""
    return {label}.union(*(ancestors(parent) for parent in graph.parents[label]))


def best_chain(label, on_route):
    """For flat predictions only, infer a path with the longest on-route prefix."""

    @lru_cache(None)
    def visit(node):
        paths = [visit(parent) + [node] for parent in sorted(graph.parents[node])]
        return (
            max(
                paths,
                key=lambda path: next(
                    (i for i, value in enumerate(path) if value not in on_route),
                    len(path),
                ),
            )
            if paths
            else [node]
        )

    return visit(label)


def gold_labels(record):
    return {normalize_label(label) for label in record["labels"]} & graph.labels.keys()


def route_nodes(gold):
    return set().union(*(ancestors(label) for label in gold))


def top1(record):
    return (
        []
        if record.get("abstained")
        else [normalize_label(label) for label in record["predictions"][:1]]
    )

## Results and paired uncertainty

**Accuracy** counts a top prediction matching any gold CWE. The Wilson interval covers example-sampling uncertainty under independence, not annotation error or adaptive tuning. Compare systems using paired differences on identical IDs; overlapping individual intervals do not answer that question.

**Hit@k** measures whether any gold CWE is retrieved; **recall@k** averages the fraction of an example's gold labels retrieved. Greedy can return only one suggestion, so its hit@5 cannot be interpreted as a five-candidate retriever. **Macro top-1 F1** includes labels seen in gold or predictions; it penalizes missing additional gold labels and is not a full multilabel classification score. **Hierarchical P/R/F1** grants partial credit by expanding predictions and labels to ancestor sets; broad shared ancestors can inflate it, so it is secondary.

In [ ]:
rows = []
for system, records in prediction_records.items():
    metrics = summary["systems"][system]
    overlap = predicted = expected = 0
    true_positive, predicted_count, gold_count = Counter(), Counter(), Counter()
    for record in records:
        gold, top = (
            gold_labels(record),
            [label for label in top1(record) if label in graph.labels],
        )
        gold_set = route_nodes(gold)
        predicted_set = ancestors(top[0]) if top else set()
        overlap += len(gold_set & predicted_set)
        predicted += len(predicted_set)
        expected += len(gold_set)
        gold_count.update(gold)
        predicted_count.update(top)
        true_positive.update(set(top) & gold)
    h_precision = overlap / predicted if predicted else 0.0
    h_recall = overlap / expected if expected else 0.0
    f1 = [
        2 * true_positive[label] / (predicted_count[label] + gold_count[label])
        for label in gold_count.keys() | predicted_count.keys()
    ]
    rows.append(
        {
            **split_system(system),
            "accuracy": metrics["accuracy"],
            "95% low": metrics["accuracy_95ci"][0],
            "95% high": metrics["accuracy_95ci"][1],
            **{
                f"hit@{k}": value
                for k, value in metrics["top_k_hit_rate"].items()
                if k > 1
            },
            **{
                f"recall@{k}": value
                for k, value in metrics["top_k_recall"].items()
                if k > 1
            },
            "abstention": metrics["abstention_rate"],
            "hier. P": h_precision,
            "hier. R": h_recall,
            "hier. F1": (
                2 * h_precision * h_recall / (h_precision + h_recall)
                if h_precision + h_recall
                else 0.0
            ),
            "macro F1": metrics["macro_f1_top1"],
        }
    )
results_table = (
    pd.DataFrame(rows)
    .set_index(["model", "method"])
    .sort_values("accuracy", ascending=False)
)
results_table.style.format(precision=3).background_gradient(
    subset=["accuracy", "hier. F1", "macro F1"], cmap="Greens"
)

In [ ]:
ordered = results_table.sort_values("accuracy")
labels = [f"{model}/{method}" for model, method in ordered.index]
ax = pd.Series(ordered["accuracy"].values, index=labels).plot.barh(
    xerr=[
        ordered["accuracy"] - ordered["95% low"],
        ordered["95% high"] - ordered["accuracy"],
    ],
    capsize=3,
    figsize=(7, 0.4 * len(labels) + 1),
    xlim=(0, 1),
    color="tab:green",
)
ax.set(
    xlabel="top-1 accuracy (bars: 95% interval)",
    title=f"{summary['examples']['evaluated']} {config['split']} examples",
);

### Per-CWE top-1 results

Precision, recall, and F1 for the most frequent gold CWEs in the sample, per system. Use this to see which weakness families each system handles well.

In [ ]:
label_rows = []
for system, records in prediction_records.items():
    for record in records:
        gold = gold_labels(record)
        top = top1(record)
        for label in gold | set(top):
            label_rows.append(
                {
                    "system": system,
                    "cwe": label,
                    "tp": int(label in gold and label in top),
                    "gold": int(label in gold),
                    "predicted": int(label in top),
                }
            )
per_label = pd.DataFrame(label_rows).groupby(["cwe", "system"]).sum()
per_label["precision"] = per_label["tp"] / per_label["predicted"].where(
    per_label["predicted"] > 0
)
per_label["recall"] = per_label["tp"] / per_label["gold"].where(per_label["gold"] > 0)
per_label["F1"] = 2 * per_label["tp"] / (per_label["gold"] + per_label["predicted"])
frequent = Counter(
    label
    for record in next(iter(prediction_records.values()))
    for label in gold_labels(record)
)
top_labels = [label for label, _ in frequent.most_common(15)]
per_cwe_f1 = per_label["F1"].unstack("system").reindex(top_labels)
per_cwe_f1.index = [
    f"{cwe_name(label)} (n={frequent[label]})" for label in per_cwe_f1.index
]
per_cwe_f1.style.format(precision=2, na_rep="—").background_gradient(
    axis=None, cmap="Greens"
)

### Cost and latency per example

Means over evaluated examples. Token counts are blank for local models, which do not report usage.

In [ ]:
cost_rows = pd.DataFrame(
    [
        {
            **split_system(system),
            "calls": record["calls"],
            "requests": record.get("requests", 0),
            "input tokens": record.get("input_tokens"),
            "output tokens": record.get("output_tokens"),
            "latency (s)": record["latency_seconds"],
        }
        for system, records in prediction_records.items()
        for record in records
    ]
)
cost_table = cost_rows.groupby(["model", "method"]).mean()
cost_table["p95 latency (s)"] = cost_rows.groupby(["model", "method"])[
    "latency (s)"
].quantile(0.95)
cost_table.style.format(precision=2, na_rep="—")

## Where does each system fail?

Each top prediction is traced along the `ChildOf` hierarchy and compared with the route(s) from a root to a gold CWE. A node is *on route* if it is a gold CWE or one of its ancestors. Greedy and beam use their recorded path; flat uses the root-to-prediction chain that stays on route longest.

- **correct**: ended on a gold CWE.
- **stopped too early**: stayed on route but stopped at an ancestor of the gold CWE.
- **went past gold**: reached the gold CWE but kept descending.
- **wrong root** / **wrong branch**: left the route at step 1 / a later step.
- **abstained**, **gold not in catalog**: no prediction, or the gold label is outside the selected view.

Use these on `dev` to decide what to change (e.g. stopping vs. routing); don't retune on `test`.

In [ ]:
OUTCOMES = (
    "correct",
    "stopped too early",
    "went past gold",
    "wrong root",
    "wrong branch",
    "abstained",
    "gold not in catalog",
)


def diagnose(record):
    """Find the first decision where the top prediction left every gold route.

    Greedy/beam use the recorded path; flat uses the root-to-prediction chain
    that stays on route longest. step is 1-based (1 = choosing a root).
    """
    gold = gold_labels(record)
    predictions = [normalize_label(label) for label in record["predictions"]]
    result = {"outcome": None, "step": None, "chose": None, "expected": [], "path": []}
    if not gold:
        return {**result, "outcome": "gold not in catalog"}
    if record.get("abstained") or not predictions:
        return {**result, "outcome": "abstained"}
    on_route = route_nodes(gold)

    def on_route_prefix(path):
        return next(
            (i for i, node in enumerate(path) if node not in on_route), len(path)
        )

    path = (
        [normalize_label(node) for node in record["paths"][0]]
        if record.get("paths")
        else best_chain(predictions[0], on_route)
    )
    result["path"] = path
    if path[-1] in gold:
        return {**result, "outcome": "correct"}
    passed = next((i for i, node in enumerate(path[:-1]) if node in gold), None)
    if passed is not None:
        return {
            **result,
            "outcome": "went past gold",
            "step": passed + 2,
            "chose": path[passed + 1],
            "expected": ["STOP"],
        }
    index = on_route_prefix(path)
    if index == len(path):
        return {
            **result,
            "outcome": "stopped too early",
            "step": index + 1,
            "chose": "STOP",
            "expected": sorted(graph.children[path[-1]] & on_route),
        }
    if index == 0:
        return {
            **result,
            "outcome": "wrong root",
            "step": 1,
            "chose": path[0],
            "expected": sorted(set(graph.roots) & on_route),
        }
    return {
        **result,
        "outcome": "wrong branch",
        "step": index + 1,
        "chose": path[index],
        "expected": sorted(graph.children[path[index - 1]] & on_route),
    }


diagnoses = pd.DataFrame(
    [
        {
            **split_system(system),
            "id": record["id"],
            **diagnose(record),
            "gold": [normalize_label(label) for label in record["labels"]],
        }
        for system, records in prediction_records.items()
        for record in records
    ]
)
outcome_table = pd.crosstab(
    [diagnoses["model"], diagnoses["method"]], diagnoses["outcome"], normalize="index"
).reindex(
    columns=[outcome for outcome in OUTCOMES if outcome in set(diagnoses["outcome"])]
)
outcome_table.style.format("{:.0%}").background_gradient(axis=None, cmap="Reds")

### Which step goes wrong?

Share of each system's examples that first go wrong at each step (step 1 = choosing a root). Correct and abstained examples are excluded.

In [ ]:
failures = diagnoses[diagnoses["step"].notna()].astype({"step": int})
step_table = (
    pd.crosstab(
        [failures["model"], failures["method"]],
        [failures["outcome"], failures["step"]],
    )
    .div(diagnoses.groupby(["model", "method"]).size(), axis=0)
    .fillna(0)
)
step_table.style.format("{:.0%}")

### Example drill-down

Failed examples with CWE names: the route taken, the step that went wrong, what was chosen, and the on-route options that should have been chosen. Filter by `model`, `method`, or `outcome` as needed.

In [ ]:
drilldown = failures.assign(
    gold=failures["gold"].map(
        lambda labels: "; ".join(
            cwe_name(label) if label in graph.labels else f"CWE-{label}"
            for label in labels
        )
    ),
    path=failures["path"].map(lambda path: " → ".join(map(cwe_name, path))),
    chose=failures["chose"].map(cwe_name),
    expected=failures["expected"].map(lambda labels: "; ".join(map(cwe_name, labels))),
)[["id", "model", "method", "gold", "outcome", "step", "path", "chose", "expected"]]
with pd.option_context("display.max_colwidth", None):
    display(drilldown.sort_values(["id", "model", "method"]).head(30))

## Decision diagnostics (descriptive)

Selector traces retain every asked question, including those on pruned branches. Calibration below is conditional on an on-route question; it excludes off-route questions and is **not end-to-end calibration**. Beam questions within one advisory are correlated; these tables and ECE bins carry no independent-trial significance claim. Recorded depth is the actual path depth at the time of the question.

An on-route choice can still be inappropriate for a report with more than one gold label. Audit examples before concluding that a stopping or routing change is beneficial.

In [ ]:
selector_systems = {
    system: records
    for system, records in prediction_records.items()
    if any(record.get("steps") for record in records)
}
decisions = []
for system, records in selector_systems.items():
    flat_method = split_system(system)["method"] == "flat"
    for record in records:
        gold = gold_labels(record)
        on_route = route_nodes(gold)
        depth_of = {None: 1}
        for path in record["paths"]:
            for depth, node in enumerate(path, start=2):
                depth_of.setdefault(node, depth)
        for step in record["steps"]:
            current, probabilities = step["current"], step["probabilities"]
            if current is not None and current not in on_route:
                continue
            if flat_method:
                correct = set(probabilities) & gold
            else:
                correct = {
                    option
                    for option in probabilities
                    if option in on_route or (option == "STOP" and current in gold)
                }
            if not correct:
                continue
            chose, confidence = min(
                probabilities.items(), key=lambda item: (-item[1], item[0])
            )
            decisions.append(
                {
                    **split_system(system),
                    "id": record["id"],
                    "depth": step.get("depth"),
                    "options": len(probabilities),
                    "chose": chose,
                    "confidence": confidence,
                    "correct": chose in correct,
                    "p(correct)": sum(probabilities[option] for option in correct),
                }
            )
decisions = pd.DataFrame(
    decisions,
    columns=[
        "model",
        "method",
        "id",
        "depth",
        "options",
        "chose",
        "confidence",
        "correct",
        "p(correct)",
    ],
)
if decisions.empty:
    print("No selector steps recorded.")
else:
    display(
        decisions.groupby(["model", "method", "depth"])
        .agg(
            decisions=("correct", "size"),
            accuracy=("correct", "mean"),
            mean_confidence=("confidence", "mean"),
            mean_p_correct=("p(correct)", "mean"),
        )
        .style.format(precision=2)
    )

In [ ]:
if not decisions.empty:
    bins = pd.cut(
        decisions["confidence"], bins=[i / 10 for i in range(11)], include_lowest=True
    )
    calibration = decisions.groupby(["model", "method", bins], observed=True).agg(
        decisions=("correct", "size"),
        accuracy=("correct", "mean"),
        confidence=("confidence", "mean"),
    )
    gap = (calibration["accuracy"] - calibration["confidence"]).abs() * calibration[
        "decisions"
    ]
    ece = (
        gap.groupby(level=["model", "method"]).sum()
        / calibration["decisions"].groupby(level=["model", "method"]).sum()
    )
    display(ece.rename("ECE").to_frame().style.format(precision=3))
    ax = None
    for (model, method), group in calibration.groupby(level=["model", "method"]):
        ax = group.plot.line(
            x="confidence",
            y="accuracy",
            marker="o",
            ax=ax,
            label=f"{model}/{method}",
            figsize=(5, 5),
        )
    ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="perfect")
    ax.set(
        xlim=(0, 1),
        ylim=(0, 1),
        xlabel="chosen-option probability",
        ylabel="decision accuracy",
        title="Selector decision calibration",
    )
    ax.legend()

### Selective accuracy by retained fraction

Sort by the recorded top routing score. These scores favor short paths and need not express actual confidence. Use `dev` to choose an operating threshold, then freeze it; the curve below is descriptive and never tunes against the test set. Coverage is the actual retained fraction after rounding.

In [ ]:
def path_probability(record):
    """Use the score actually returned, including flat selectors and DAG pruning."""
    return record.get("scores", [None])[0] if record.get("scores") else None


coverage_rows = []
for system, records in selector_systems.items():
    scored = sorted(
        (
            (
                path_probability(record),
                bool(
                    gold_labels(record)
                    & {normalize_label(label) for label in record["predictions"][:1]}
                ),
            )
            for record in records
        ),
        key=lambda item: -(item[0] or 0),
    )
    for fraction in (0.1, 0.25, 0.5, 0.75, 1.0):
        kept = scored[: max(1, round(fraction * len(scored)))]
        coverage_rows.append(
            {
                **split_system(system),
                "coverage": len(kept) / len(scored),
                "accuracy": sum(correct for _, correct in kept) / len(kept),
                "min confidence": kept[-1][0],
            }
        )
coverage = pd.DataFrame(
    coverage_rows, columns=["model", "method", "coverage", "accuracy", "min confidence"]
)
if coverage.empty:
    print("No selector systems to analyze.")
else:
    curve = coverage.pivot_table(
        index="coverage", columns=["model", "method"], values="accuracy"
    )
    curve.plot(
        marker="o",
        figsize=(7, 4),
        ylim=(0, 1),
        ylabel="top-1 accuracy",
        title="Accuracy when keeping the most confident examples",
    )
    display(curve.style.format(precision=3))

## Paired top-1 differences

Bootstrap **advisory IDs jointly**, not each system independently. The interval describes this matched sample under an independent-advisory assumption; package families may remain correlated. Preselect a primary contrast before a confirmation run. Looking at many contrasts requires multiple-comparison control or an untouched final holdout.

In [ ]:
import random


def paired_accuracy_difference(first, second, *, seed=0, resamples=2000):
    """Return second minus first accuracy and its paired percentile interval."""
    left, right = ({row["id"]: row for row in rows} for rows in (first, second))
    if not left or left.keys() != right.keys():
        raise ValueError("Paired systems must contain identical, nonempty IDs")
    if len(left) != len(first) or len(right) != len(second):
        raise ValueError("Paired IDs must be unique")
    deltas = []
    for identifier in sorted(left):
        a, b = left[identifier], right[identifier]
        if gold_labels(a) != gold_labels(b):
            raise ValueError("Paired systems disagree on gold labels")
        deltas.append(
            int(bool(set(top1(b)) & gold_labels(b)))
            - int(bool(set(top1(a)) & gold_labels(a)))
        )
    rng, size = random.Random(seed), len(deltas)
    samples = sorted(sum(rng.choices(deltas, k=size)) / size for _ in range(resamples))
    return {
        "n": size,
        "accuracy_delta": sum(deltas) / size,
        "paired_95ci": [
            samples[int(0.025 * resamples)],
            samples[int(0.975 * resamples)],
        ],
    }


systems = list(prediction_records)
if len(systems) >= 2:
    comparison = paired_accuracy_difference(
        prediction_records[systems[0]], prediction_records[systems[1]]
    )
    display({"first": systems[0], "second": systems[1], **comparison})

## Interpretation and next action

Use failures to distinguish root routing, wrong branches, and overspecific stopping; inspect per-label support before generalizing. Keep configuration, `metrics.json`, JSONL predictions, checkpoints and dataset report together. Main notebook outputs are cleared before commit to avoid stale claims and advisory disclosure.

The executable pipeline is a protocol, not a completed validation of OpenAI quality. See `AUDIT.md` for prioritized harness improvements and `hypotheses.ipynb` for alternative prompts, roots, retrieval and calibration studies. Promote a hypothesis only after a matched dev experiment improves a predeclared metric at an acceptable request budget.